In [ ]:
import numpy as np
from netCDF4 import Dataset
from scipy.io import loadmat
from scipy.ndimage import distance_transform_edt
from matplotlib import pyplot as plt

#X, Y = np.meshgrid(np.arange(-720e3,960e3,1e3), np.arange(-3450e3,-570e3,1e3))


In [ ]:
bedmachine_ds = Dataset('/home/theghub/denisfelikson/projects/grisoceantofjords/files/BedMachineGreenland/BedMachineGreenland-v5.nc')
x_bed = bedmachine_ds['x'][:]
y_bed = bedmachine_ds['y'][:]
BED = bedmachine_ds['bed'][:,:]
mask = bedmachine_ds['mask'][:,:]
bedmachine_ds.close()


In [ ]:
data = loadmat('/home/theghub/denisfelikson/projects/grisoceantofjords/files/BedMachineGreenland/ocean_extrap_MIROC5_RCP85.mat')
data.keys()
S = data['S']
T = data['T']
basins = data['basins'][0]
year = data['year'][0]
z = data['z'][0]


In [ ]:
# Define open ocean pixels
FAR = (mask != 0)
FRAME = np.zeros(FAR.shape)
FRAME[0,:]  = 1
FRAME[-1,:] = 1
FRAME[:,0]  = 1
FRAME[:,-1] = 1

FAR = np.logical_or(1e3*distance_transform_edt(FAR) > 200e3, FRAME)


In [ ]:
from skimage.measure import label#, regionprops, regionprops_table

# Initialize output
connectivity = np.full( (BED.shape[0], BED.shape[1], z.shape[0]), False, dtype=bool)

for i in range(len(z)):
    print('   -- Depth = ' + str(z[i]) + ' ' + str(i) + '/' + str(len(z)))
    depth = z[i]
    A = (BED<=depth)
    print('    ... Labelling')
    CC = label(A)
    print('    ... Finding unique labels')
    pos = (FAR) & (BED<depth)
    list = np.unique(CC[pos])
    connectivity[:,:,i] = np.isin(CC,list)
    break
    

In [ ]:
connectivity.shape